# 🚦 UK Road Casualty Statistics 2025 — Exploratory Data Analysis

**Dataset**: Department for Transport (DfT) — Provisional Road Casualty Statistics 2025

| File | Rows | Columns |
|---|---|---|
| Casualty | 60,991 | 23 |
| Collision | 48,472 | 44 |
| Vehicle | 87,805 | 32 |

**Libraries**: Pandas · NumPy · Matplotlib · Seaborn

---
### 📌 Analysis Sections
1. Imports & Global Style
2. Data Loading & Overview
3. Data Cleaning & Preprocessing
4. Helper Functions
5. Severity Analysis
6. Temporal Patterns
7. Road & Environmental Conditions
8. Casualty Demographics
9. Vehicle Analysis
10. Geographic & Multi-Variable Analysis
11. Correlation Heatmap
12. Key Findings Summary

---
## 1. Imports & Global Style

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Dark background style (matching reference project) ──────────────────────
plt.style.use('dark_background')

DARK_BG   = '#1a1a2e'
PANEL_BG  = '#16213e'
GREEN     = '#00d26a'
BLUE      = '#4da6ff'
ORANGE    = '#ff9f40'
RED       = '#ff6b6b'
PURPLE    = '#c77dff'
YELLOW    = '#ffd700'
TEAL      = '#00b4d8'

PALETTE5  = [GREEN, BLUE, ORANGE, RED, PURPLE]
PALETTE3  = [GREEN, BLUE, ORANGE]

plt.rcParams.update({
    'figure.facecolor':  DARK_BG,
    'axes.facecolor':    PANEL_BG,
    'axes.edgecolor':    '#444466',
    'axes.labelcolor':   'white',
    'xtick.color':       'white',
    'ytick.color':       'white',
    'text.color':        'white',
    'grid.color':        '#2a2a4a',
    'grid.linestyle':    '--',
    'grid.alpha':        0.5,
    'figure.dpi':        130,
    'font.size':         11,
})

print("✅ Imports & style configured")

---
## 2. Data Loading & Overview

In [ ]:
BASE = 'dft-road-casualty-statistics-{}-provisional-2025.csv'

casualty  = pd.read_csv(BASE.format('casualty'),   low_memory=False)
collision = pd.read_csv(BASE.format('collision'),  low_memory=False)
vehicle   = pd.read_csv(BASE.format('vehicle'),    low_memory=False)

for name, df in [('Casualty', casualty), ('Collision', collision), ('Vehicle', vehicle)]:
    print(f"📋 {name:12s} — {df.shape[0]:>6,} rows × {df.shape[1]:>2} cols")

In [ ]:
print("=== COLLISION sample ===")
display(collision.head(3))
print("\n=== CASUALTY sample ===")
display(casualty.head(3))
print("\n=== VEHICLE sample ===")
display(vehicle.head(3))

In [ ]:
# ── Missing Value Overview ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('Missing Value Analysis Across Datasets', fontsize=15, fontweight='bold', color='white', y=1.02)

for ax, (name, df) in zip(axes, [('Casualty', casualty), ('Collision', collision), ('Vehicle', vehicle)]):
    miss = (df.isnull().mean() * 100).sort_values(ascending=False)
    miss = miss[miss > 0]
    ax.set_facecolor(PANEL_BG)
    if miss.empty:
        ax.text(0.5, 0.5, '✅ No Missing Values', ha='center', va='center',
                transform=ax.transAxes, fontsize=13, color=GREEN, fontweight='bold')
    else:
        bars = ax.barh(miss.index, miss.values, color=ORANGE, edgecolor='none')
        for bar, val in zip(bars, miss.values):
            ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
                    f'{val:.2f}%', va='center', fontsize=9, color='white')
        ax.set_xlabel('% Missing')
    ax.set_title(f'{name}', fontsize=12, fontweight='bold', color=YELLOW)
    ax.grid(True, axis='x')

plt.tight_layout()
plt.savefig('01_missing_values.png', bbox_inches='tight', facecolor=DARK_BG)
plt.show()

---
## 3. Data Cleaning & Preprocessing

In [ ]:
# ── Label maps ─────────────────────────────────────────────────────────────
SEVERITY_MAP = {1: 'Fatal', 2: 'Serious', 3: 'Slight'}
SEX_MAP      = {1: 'Male', 2: 'Female', -1: 'Unknown', 9: 'Unknown'}
DAY_MAP      = {1:'Sun', 2:'Mon', 3:'Tue', 4:'Wed', 5:'Thu', 6:'Fri', 7:'Sat'}
WEATHER_MAP  = {
    1:'Fine/No wind', 2:'Raining/No wind', 3:'Snowing/No wind',
    4:'Fine/High wind', 5:'Raining/High wind', 6:'Fog or mist',
    7:'Other', 8:'Unknown', 9:'Unknown'
}
LIGHT_MAP    = {
    1:'Daylight', 4:'Darkness – lit', 5:'Darkness – unlit',
    6:'Darkness – no light', 7:'Darkness – unknown'
}
ROAD_COND    = {1:'Dry', 2:'Wet/Damp', 3:'Snow', 4:'Frost/Ice',
                5:'Flood', 6:'Oil/Diesel', 7:'Mud', -1:'Unknown'}
URBAN_MAP    = {1:'Urban', 2:'Rural', 3:'Unallocated'}
VEH_TYPE_MAP = {
    1:'Pedal cycle', 2:'Motorcycle <50cc', 3:'Motorcycle 50-125cc',
    4:'Motorcycle 125-500cc', 5:'Motorcycle >500cc', 8:'Taxi/Private',
    9:'Car', 10:'Minibus', 11:'Bus/Coach', 16:'Horse/Rider',
    17:'Agricultural', 18:'Tram', 19:'Van/Goods ≤3.5t',
    20:'Goods 3.5-7.5t', 21:'Goods >7.5t', 22:'Mobility scooter',
    23:'E-Motorcycle', 90:'Other', 97:'Motorcycle unknown', 98:'Unknown'
}
CAS_TYPE_MAP = {
    0:'Pedestrian', 1:'Cyclist', 2:'Motorcycle <50cc', 3:'Motorcycle 50-125cc',
    4:'Motorcycle 125-500cc', 5:'Motorcycle >500cc', 8:'Taxi/Private',
    9:'Car occupant', 10:'Minibus', 11:'Bus/Coach', 16:'Horse rider',
    17:'Agricultural', 18:'Tram', 19:'Van/Goods ≤3.5t',
    20:'Goods 3.5-7.5t', 21:'Goods >7.5t', 22:'Mobility scooter',
    23:'E-Motorcycle', 90:'Other', 97:'Motorcycle unknown', 98:'Unknown'
}
SPEED_MAP = {20:20, 30:30, 40:40, 50:50, 60:60, 70:70}
AGE_BAND  = {
    1:'0-5', 2:'6-10', 3:'11-15', 4:'16-20', 5:'21-25',
    6:'26-35', 7:'36-45', 8:'46-55', 9:'56-65', 10:'66-75', 11:'76+'
}

# ── Apply labels ────────────────────────────────────────────────────────────
collision['severity_label']  = collision['collision_severity'].map(SEVERITY_MAP)
collision['day_label']       = collision['day_of_week'].map(DAY_MAP)
collision['weather_label']   = collision['weather_conditions'].map(WEATHER_MAP).fillna('Other')
collision['light_label']     = collision['light_conditions'].map(LIGHT_MAP).fillna('Other')
collision['road_cond_label'] = collision['road_surface_conditions'].map(ROAD_COND).fillna('Unknown')
collision['urban_label']     = collision['urban_or_rural_area'].map(URBAN_MAP).fillna('Unknown')

# ── Parse date/time ─────────────────────────────────────────────────────────
collision['date_parsed'] = pd.to_datetime(collision['date'], errors='coerce')
collision['month']       = collision['date_parsed'].dt.month
collision['month_name']  = collision['date_parsed'].dt.strftime('%b')
collision['hour']        = pd.to_datetime(collision['time'], format='%H:%M', errors='coerce').dt.hour

# ── Casualty labels ─────────────────────────────────────────────────────────
casualty['severity_label'] = casualty['casualty_severity'].map(SEVERITY_MAP)
casualty['sex_label']      = casualty['sex_of_casualty'].map(SEX_MAP).fillna('Unknown')
casualty['cas_type_label'] = casualty['casualty_type'].map(CAS_TYPE_MAP).fillna('Other')
casualty['age_band_label'] = casualty['age_band_of_casualty'].map(AGE_BAND).fillna('Unknown')

# ── Vehicle labels ───────────────────────────────────────────────────────────
vehicle['veh_type_label'] = vehicle['vehicle_type'].map(VEH_TYPE_MAP).fillna('Other')
vehicle['sex_driver']     = vehicle['sex_of_driver'].map(SEX_MAP).fillna('Unknown')

print("✅ Label mapping complete")
print(f"   collision:  {collision.shape}")
print(f"   casualty:   {casualty.shape}")
print(f"   vehicle:    {vehicle.shape}")

---
## 4. Helper Functions

In [ ]:
def add_bar_labels(ax, bars, fmt='{:.0f}', color='white', fontsize=9, padding=0.5):
    """Add value labels on top of / beside bar charts."""
    for bar in bars:
        w = bar.get_width()
        h = bar.get_height()
        if bar.get_width() > 0 and bar.get_height() < bar.get_width():  # horizontal
            ax.text(w + padding, bar.get_y() + h/2,
                    fmt.format(w), va='center', ha='left', fontsize=fontsize, color=color)
        else:  # vertical
            ax.text(bar.get_x() + bar.get_width()/2, h + padding,
                    fmt.format(h), ha='center', va='bottom', fontsize=fontsize, color=color)

def style_ax(ax, title='', xlabel='', ylabel='', title_color=YELLOW):
    """Apply consistent dark-theme styling to an axes object."""
    ax.set_facecolor(PANEL_BG)
    ax.set_title(title, fontsize=12, fontweight='bold', color=title_color, pad=8)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.tick_params(colors='white')
    ax.grid(True, alpha=0.3)
    for spine in ax.spines.values():
        spine.set_edgecolor('#333355')

def severity_colors(labels):
    """Return color per severity label."""
    cmap = {'Fatal': RED, 'Serious': ORANGE, 'Slight': GREEN}
    return [cmap.get(l, BLUE) for l in labels]

def get_top_n(series, n=10):
    """Value counts, top-n, percentage."""
    vc = series.value_counts().head(n)
    pct = (vc / vc.sum() * 100).round(1)
    return vc, pct

print("✅ Helper functions defined")

---
## 5. Severity Analysis

In [ ]:
# ── 5A: Severity overview ───────────────────────────────────────────────────
sev_col  = collision['severity_label'].value_counts().reindex(['Fatal','Serious','Slight'])
sev_cas  = casualty['severity_label'].value_counts().reindex(['Fatal','Serious','Slight'])
cas_sev_type = casualty.groupby(['cas_type_label','severity_label']).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('Severity Analysis', fontsize=16, fontweight='bold', color='white', y=1.01)

# ── Panel 1: Collision severity pie ─────────────────────────────────────────
ax = axes[0]
ax.set_facecolor(PANEL_BG)
colors_sev = [RED, ORANGE, GREEN]
wedges, texts, autotexts = ax.pie(
    sev_col.values, labels=sev_col.index, autopct='%1.1f%%',
    colors=colors_sev, startangle=140,
    wedgeprops={'edgecolor': DARK_BG, 'linewidth': 2},
    textprops={'color': 'white', 'fontsize': 11}
)
for at in autotexts:
    at.set_color('white'); at.set_fontweight('bold'); at.set_fontsize(10)
ax.set_title('Collisions by Severity', fontsize=12, fontweight='bold', color=YELLOW)

# ── Panel 2: Casualty severity bar ──────────────────────────────────────────
ax = axes[1]
bars = ax.bar(sev_cas.index, sev_cas.values, color=[RED, ORANGE, GREEN], edgecolor='none', width=0.6)
for bar, val in zip(bars, sev_cas.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{val:,}', ha='center', va='bottom', color='white', fontsize=11, fontweight='bold')
style_ax(ax, 'Total Casualties by Severity', 'Severity', 'Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# ── Panel 3: Casualty type × severity stacked bar (top 8 types) ─────────────
ax = axes[2]
top_types = cas_sev_type.sum(axis=1).nlargest(8).index
df_plot = cas_sev_type.loc[top_types]
bot = np.zeros(len(df_plot))
for col, col_color in zip(['Fatal','Serious','Slight'], [RED, ORANGE, GREEN]):
    if col in df_plot.columns:
        vals = df_plot[col].values
        ax.barh(df_plot.index, vals, left=bot, color=col_color, label=col, alpha=0.9)
        bot += vals
ax.legend(loc='lower right', fontsize=9)
style_ax(ax, 'Casualty Type × Severity (Top 8)', 'Count', '')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('02_severity_analysis.png', bbox_inches='tight', facecolor=DARK_BG)
plt.show()

# Print stats
print(f"Total Collisions : {len(collision):,}")
print(f"Total Casualties : {len(casualty):,}")
print(f"Total Vehicles   : {len(vehicle):,}")
print()
print("Collision Severity:")
for sev in ['Fatal','Serious','Slight']:
    n = sev_col.get(sev, 0)
    print(f"  {sev:10s}: {n:>6,}  ({n/len(collision)*100:.1f}%)")

---
## 6. Temporal Patterns

In [ ]:
# ── 6A: Day of week × hour heatmap + monthly trend ─────────────────────────
day_order  = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
month_abbr = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

collision['day_label'] = pd.Categorical(collision['day_label'], categories=day_order, ordered=True)
monthly = collision.groupby('month').size()

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('Temporal Patterns', fontsize=16, fontweight='bold', color='white', y=1.01)

# ── Panel 1: Collisions by day ───────────────────────────────────────────────
ax = axes[0]
day_counts = collision['day_label'].value_counts().reindex(day_order)
bar_colors = [RED if d in ['Fri','Sat'] else BLUE for d in day_order]
bars = ax.bar(day_order, day_counts.values, color=bar_colors, edgecolor='none', width=0.7)
for bar, val in zip(bars, day_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{val:,}', ha='center', va='bottom', color='white', fontsize=9)
style_ax(ax, 'Collisions by Day of Week', 'Day', 'Count')

# ── Panel 2: Collisions by hour ──────────────────────────────────────────────
ax = axes[1]
hour_counts = collision['hour'].value_counts().sort_index()
peak_hours  = hour_counts.nlargest(3).index.tolist()
h_colors    = [RED if h in peak_hours else TEAL for h in hour_counts.index]
ax.bar(hour_counts.index, hour_counts.values, color=h_colors, edgecolor='none', width=0.8)
ax.axvspan(7, 9, alpha=0.12, color=ORANGE, label='Morning peak')
ax.axvspan(16, 19, alpha=0.12, color=RED, label='Evening peak')
ax.legend(fontsize=9)
style_ax(ax, 'Collisions by Hour of Day', 'Hour', 'Count')

# ── Panel 3: Monthly collision count ────────────────────────────────────────
ax = axes[2]
valid_months = monthly.reindex(range(1, 13), fill_value=0)
m_labels = [month_abbr[m-1] for m in valid_months.index]
bars = ax.bar(m_labels, valid_months.values, color=GREEN, edgecolor='none', width=0.7)
ax.plot(m_labels, valid_months.values, color=YELLOW, marker='o', linewidth=2, zorder=5)
style_ax(ax, 'Monthly Collision Count', 'Month', 'Count')

plt.tight_layout()
plt.savefig('03_temporal_patterns.png', bbox_inches='tight', facecolor=DARK_BG)
plt.show()

In [ ]:
# ── 6B: Day × Hour heatmap ──────────────────────────────────────────────────
hmap = collision.dropna(subset=['hour','day_label'])
pivot = hmap.pivot_table(index='day_label', columns='hour', values='collision_index',
                         aggfunc='count', fill_value=0)
pivot = pivot.reindex(day_order)

fig, ax = plt.subplots(figsize=(18, 5))
fig.patch.set_facecolor(DARK_BG)
ax.set_facecolor(PANEL_BG)
sns.heatmap(pivot, ax=ax, cmap='YlOrRd', linewidths=0.3, linecolor='#1a1a2e',
            annot=True, fmt='d', annot_kws={'size': 8},
            cbar_kws={'label': 'Collisions'})
ax.set_title('Collision Heatmap — Day of Week × Hour', fontsize=14, fontweight='bold', color='white', pad=10)
ax.set_xlabel('Hour of Day', fontsize=11)
ax.set_ylabel('Day of Week', fontsize=11)
ax.tick_params(colors='white')

plt.tight_layout()
plt.savefig('04_day_hour_heatmap.png', bbox_inches='tight', facecolor=DARK_BG)
plt.show()

---
## 7. Road & Environmental Conditions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 12))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('Road & Environmental Conditions Analysis', fontsize=16, fontweight='bold', color='white', y=1.01)

# ── Panel 1: Weather conditions ──────────────────────────────────────────────
ax = axes[0][0]
wc = collision['weather_label'].value_counts().head(7)
bars = ax.barh(wc.index[::-1], wc.values[::-1], color=BLUE, edgecolor='none')
for bar in bars:
    w = bar.get_width()
    ax.text(w + 20, bar.get_y() + bar.get_height()/2,
            f'{w:,}  ({w/len(collision)*100:.1f}%)', va='center', color='white', fontsize=9)
style_ax(ax, 'Collisions by Weather Condition', 'Count', '')
ax.set_xlim(0, wc.max() * 1.3)

# ── Panel 2: Road surface ─────────────────────────────────────────────────────
ax = axes[0][1]
rc = collision['road_cond_label'].value_counts()
wedge_colors = [GREEN, BLUE, ORANGE, RED, PURPLE, YELLOW, TEAL]
wedges, texts, autotexts = ax.pie(
    rc.values, labels=rc.index, autopct='%1.1f%%',
    colors=wedge_colors[:len(rc)], startangle=90,
    wedgeprops={'edgecolor': DARK_BG, 'linewidth': 2},
    textprops={'color': 'white', 'fontsize': 9}
)
for at in autotexts:
    at.set_color('white'); at.set_fontsize(9)
style_ax(ax, 'Road Surface Conditions', '')

# ── Panel 3: Light conditions × severity ─────────────────────────────────────
ax = axes[1][0]
lc = collision.groupby(['light_label','severity_label']).size().unstack(fill_value=0)
lc = lc.reindex(columns=['Fatal','Serious','Slight'], fill_value=0)
lc_sorted = lc.sum(axis=1).sort_values(ascending=True)
lc = lc.loc[lc_sorted.index]
bot = np.zeros(len(lc))
for col, col_color in zip(['Fatal','Serious','Slight'], [RED, ORANGE, GREEN]):
    ax.barh(lc.index, lc[col], left=bot, color=col_color, label=col, alpha=0.9)
    bot += lc[col].values
ax.legend(fontsize=9, loc='lower right')
style_ax(ax, 'Light Conditions × Severity', 'Count', '')

# ── Panel 4: Speed limit distribution ────────────────────────────────────────
ax = axes[1][1]
speed_sev = collision.groupby(['speed_limit','severity_label']).size().unstack(fill_value=0)
speed_sev = speed_sev.reindex(columns=['Fatal','Serious','Slight'], fill_value=0)
valid_speeds = [20, 30, 40, 50, 60, 70]
speed_sev = speed_sev.loc[speed_sev.index.isin(valid_speeds)].reindex(valid_speeds, fill_value=0)
x = np.arange(len(speed_sev))
w = 0.28
for i, (col, col_color) in enumerate(zip(['Fatal','Serious','Slight'], [RED, ORANGE, GREEN])):
    bars = ax.bar(x + i*w, speed_sev[col], width=w, color=col_color, label=col, alpha=0.9)
ax.set_xticks(x + w)
ax.set_xticklabels([f'{s} mph' for s in valid_speeds])
ax.legend(fontsize=9)
style_ax(ax, 'Collisions by Speed Limit × Severity', 'Speed Limit', 'Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('05_road_conditions.png', bbox_inches='tight', facecolor=DARK_BG)
plt.show()

---
## 8. Casualty Demographics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 12))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('Casualty Demographics Analysis', fontsize=16, fontweight='bold', color='white', y=1.01)

# ── Panel 1: Sex of casualty ─────────────────────────────────────────────────
ax = axes[0][0]
sex = casualty['sex_label'].value_counts()
sex = sex[sex.index.isin(['Male','Female'])]
bars = ax.bar(sex.index, sex.values, color=[BLUE, PURPLE], edgecolor='none', width=0.5)
for bar, val in zip(bars, sex.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}\n({val/sex.sum()*100:.1f}%)', ha='center', va='bottom',
            color='white', fontsize=11, fontweight='bold')
style_ax(ax, 'Casualties by Sex', 'Sex', 'Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# ── Panel 2: Age band distribution ───────────────────────────────────────────
ax = axes[0][1]
age_order = ['0-5','6-10','11-15','16-20','21-25','26-35','36-45','46-55','56-65','66-75','76+']
age_sev = casualty.groupby(['age_band_label','severity_label']).size().unstack(fill_value=0)
age_sev = age_sev.reindex(age_order).fillna(0)
age_sev = age_sev.reindex(columns=['Fatal','Serious','Slight'], fill_value=0)
bot = np.zeros(len(age_sev))
for col, col_color in enumerate(['Fatal','Serious','Slight']):
    colors_list = [RED, ORANGE, GREEN]
    ax.bar(age_sev.index, age_sev[col_color], bottom=bot, color=colors_list[col], label=col_color, alpha=0.9)
    bot += age_sev[col_color].values
ax.legend(fontsize=9)
ax.set_xticklabels(age_order, rotation=45, ha='right', fontsize=9)
style_ax(ax, 'Casualties by Age Band × Severity', 'Age Band', 'Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# ── Panel 3: Top casualty types ──────────────────────────────────────────────
ax = axes[1][0]
ct = casualty['cas_type_label'].value_counts().head(10)
bar_colors_ct = [GREEN if i == 0 else BLUE if i < 3 else TEAL for i in range(len(ct))]
bars = ax.barh(ct.index[::-1], ct.values[::-1], color=bar_colors_ct[::-1], edgecolor='none')
for bar in bars:
    w = bar.get_width()
    ax.text(w + 30, bar.get_y() + bar.get_height()/2,
            f'{w:,}  ({w/len(casualty)*100:.1f}%)', va='center', color='white', fontsize=9)
style_ax(ax, 'Top 10 Casualty Types', 'Count', '')
ax.set_xlim(0, ct.max() * 1.3)

# ── Panel 4: Severity × sex breakdown ────────────────────────────────────────
ax = axes[1][1]
sex_sev = casualty[casualty['sex_label'].isin(['Male','Female'])]    .groupby(['sex_label','severity_label']).size().unstack(fill_value=0)    .reindex(columns=['Fatal','Serious','Slight'], fill_value=0)
x = np.arange(2)
w = 0.28
for i, (col, col_color) in enumerate(zip(['Fatal','Serious','Slight'], [RED, ORANGE, GREEN])):
    bars = ax.bar(x + i*w, sex_sev[col], width=w, color=col_color, label=col, alpha=0.9)
    for bar, val in zip(bars, sex_sev[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                str(val), ha='center', va='bottom', color='white', fontsize=9)
ax.set_xticks(x + w)
ax.set_xticklabels(['Female', 'Male'])
ax.legend(fontsize=9)
style_ax(ax, 'Severity × Sex', 'Sex', 'Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('06_casualty_demographics.png', bbox_inches='tight', facecolor=DARK_BG)
plt.show()

---
## 9. Vehicle Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 12))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('Vehicle Involvement Analysis', fontsize=16, fontweight='bold', color='white', y=1.01)

# ── Panel 1: Top vehicle types ───────────────────────────────────────────────
ax = axes[0][0]
vt = vehicle['veh_type_label'].value_counts().head(12)
bar_colors_v = [GREEN if i == 0 else BLUE if i < 4 else TEAL for i in range(len(vt))]
bars = ax.barh(vt.index[::-1], vt.values[::-1], color=bar_colors_v[::-1], edgecolor='none')
for bar in bars:
    w = bar.get_width()
    ax.text(w + 100, bar.get_y() + bar.get_height()/2,
            f'{w:,}', va='center', color='white', fontsize=9)
style_ax(ax, 'Top 12 Vehicle Types Involved', 'Count', '')
ax.set_xlim(0, vt.max() * 1.2)

# ── Panel 2: Driver sex ──────────────────────────────────────────────────────
ax = axes[0][1]
ds = vehicle['sex_driver'].value_counts()
ds = ds[ds.index.isin(['Male','Female'])]
wedges, texts, autotexts = ax.pie(
    ds.values, labels=ds.index, autopct='%1.1f%%',
    colors=[BLUE, PURPLE], startangle=90,
    wedgeprops={'edgecolor': DARK_BG, 'linewidth': 2},
    textprops={'color': 'white', 'fontsize': 12}
)
for at in autotexts:
    at.set_color('white'); at.set_fontweight('bold')
style_ax(ax, 'Involved Drivers by Sex', '')

# ── Panel 3: Driver age band distribution ────────────────────────────────────
ax = axes[1][0]
AGE_BAND_DRV = {
    1:'16-20', 2:'21-25', 3:'26-35', 4:'36-45',
    5:'46-55', 6:'56-65', 7:'66-75', 8:'76+'
}
vehicle['driver_age_band_label'] = vehicle['age_band_of_driver'].map(AGE_BAND_DRV).fillna('Unknown')
age_drv_order = ['16-20','21-25','26-35','36-45','46-55','56-65','66-75','76+']
drv_age = vehicle[vehicle['driver_age_band_label'].isin(age_drv_order)]    ['driver_age_band_label'].value_counts().reindex(age_drv_order, fill_value=0)
bar_c = [RED if a in ['16-20','21-25'] else BLUE for a in age_drv_order]
bars = ax.bar(age_drv_order, drv_age.values, color=bar_c, edgecolor='none', width=0.7)
for bar, val in zip(bars, drv_age.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{val:,}', ha='center', va='bottom', color='white', fontsize=9)
ax.set_xticklabels(age_drv_order, rotation=30, ha='right')
style_ax(ax, 'Driver Age Band Distribution', 'Age Band', 'Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# ── Panel 4: Vehicles per collision distribution ──────────────────────────────
ax = axes[1][1]
vpc = collision['number_of_vehicles'].value_counts().sort_index().head(6)
bars = ax.bar(vpc.index.astype(str), vpc.values, color=ORANGE, edgecolor='none', width=0.6)
for bar, val in zip(bars, vpc.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}\n({val/len(collision)*100:.1f}%)', ha='center', va='bottom',
            color='white', fontsize=10, fontweight='bold')
style_ax(ax, 'Vehicles per Collision', 'Number of Vehicles', 'Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('07_vehicle_analysis.png', bbox_inches='tight', facecolor=DARK_BG)
plt.show()

---
## 10. Geographic & Multi-Variable Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('Geographic & Multi-Variable Analysis', fontsize=16, fontweight='bold', color='white', y=1.01)

# ── Panel 1: Urban vs Rural pie ──────────────────────────────────────────────
ax = axes[0]
ur = collision['urban_label'].value_counts()
ur = ur[ur.index.isin(['Urban','Rural'])]
wedges, texts, autotexts = ax.pie(
    ur.values, labels=ur.index, autopct='%1.1f%%',
    colors=[TEAL, ORANGE], startangle=140,
    wedgeprops={'edgecolor': DARK_BG, 'linewidth': 3},
    textprops={'color': 'white', 'fontsize': 13}
)
for at in autotexts:
    at.set_color('white'); at.set_fontweight('bold'); at.set_fontsize(12)
style_ax(ax, 'Urban vs Rural Collisions', '')

# ── Panel 2: Casualties per collision distribution ───────────────────────────
ax = axes[1]
cpc = collision['number_of_casualties'].value_counts().sort_index().head(6)
bars = ax.bar(cpc.index.astype(str), cpc.values, color=PURPLE, edgecolor='none', width=0.6)
for bar, val in zip(bars, cpc.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
            f'{val:,}', ha='center', va='bottom', color='white', fontsize=10, fontweight='bold')
style_ax(ax, 'Casualties per Collision', 'Number of Casualties', 'Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# ── Panel 3: Scatter — vehicles vs casualties (by severity) ──────────────────
ax = axes[2]
sev_colors_map = {'Fatal': RED, 'Serious': ORANGE, 'Slight': GREEN}
sample = collision.dropna(subset=['severity_label']).sample(min(3000, len(collision)), random_state=42)
for sev, grp in sample.groupby('severity_label'):
    ax.scatter(grp['number_of_vehicles'], grp['number_of_casualties'],
               alpha=0.35, s=25, color=sev_colors_map.get(sev, BLUE), label=sev)
ax.set_xlim(0.5, 6.5)
ax.set_ylim(0.5, 8.5)
ax.legend(fontsize=10)
style_ax(ax, 'Vehicles vs Casualties (by Severity)', 'Number of Vehicles', 'Number of Casualties')

plt.tight_layout()
plt.savefig('08_geographic_multivar.png', bbox_inches='tight', facecolor=DARK_BG)
plt.show()

In [ ]:
# ── Multi-Metric Radar: Severity × Road/Time Factors ───────────────────────
from matplotlib.patches import FancyArrowPatch

def radar_chart(ax, values_dict, categories, title):
    N = len(categories)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_thetagrids(np.degrees(angles[:-1]), categories, fontsize=9, color='white')
    ax.set_facecolor(PANEL_BG)
    colors_r = [RED, ORANGE, GREEN]
    for (label, vals), col in zip(values_dict.items(), colors_r):
        v = vals + vals[:1]
        ax.plot(angles, v, 'o-', linewidth=2, label=label, color=col)
        ax.fill(angles, v, alpha=0.15, color=col)
    ax.set_ylim(0, 1)
    ax.set_title(title, fontsize=12, fontweight='bold', color=YELLOW, pad=18)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
    ax.tick_params(colors='white')
    ax.grid(color='#2a2a4a', alpha=0.6)

# Build metrics per severity
metrics = {}
for sev in ['Fatal', 'Serious', 'Slight']:
    sub = collision[collision['severity_label'] == sev]
    n   = len(sub) + 1e-9
    metrics[sev] = [
        sub['hour'].between(20, 23).sum() / n,                     # Night (20-23h)
        (sub['weather_label'] != 'Fine/No wind').sum() / n,        # Bad weather
        (sub['road_cond_label'] != 'Dry').sum() / n,               # Wet/Icy road
        (sub['urban_label'] == 'Rural').sum() / n,                 # Rural area
        sub['number_of_vehicles'].gt(1).sum() / n,                 # Multi-vehicle
    ]

cats = ['Night\n(20-23h)', 'Bad\nWeather', 'Wet/Icy\nRoad', 'Rural\nArea', 'Multi-\nVehicle']

fig = plt.figure(figsize=(10, 7))
fig.patch.set_facecolor(DARK_BG)
ax = fig.add_subplot(111, polar=True)
radar_chart(ax, metrics, cats, 'Multi-Metric Radar: Severity × Risk Factors')

plt.tight_layout()
plt.savefig('09_radar_chart.png', bbox_inches='tight', facecolor=DARK_BG)
plt.show()

---
## 11. Correlation Heatmap

In [ ]:
num_cols = ['collision_severity', 'number_of_vehicles', 'number_of_casualties',
            'speed_limit', 'light_conditions', 'weather_conditions',
            'road_surface_conditions', 'urban_or_rural_area', 'hour', 'month']

corr_df = collision[num_cols].dropna()
corr    = corr_df.corr()

fig, ax = plt.subplots(figsize=(13, 10))
fig.patch.set_facecolor(DARK_BG)
ax.set_facecolor(PANEL_BG)

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax, mask=mask, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', annot_kws={'size': 10},
            linewidths=0.5, linecolor='#1a1a2e',
            cbar_kws={'label': 'Pearson r', 'shrink': 0.8})

ax.set_title('Correlation Heatmap — Collision Numeric Features',
             fontsize=14, fontweight='bold', color='white', pad=14)
ax.tick_params(colors='white', labelsize=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')

plt.tight_layout()
plt.savefig('10_correlation_heatmap.png', bbox_inches='tight', facecolor=DARK_BG)
plt.show()

---
## 12. Key Findings Summary

In [ ]:
print("="*65)
print(" 🚦 UK ROAD CASUALTY STATISTICS 2025 — KEY FINDINGS")
print("="*65)

total_col = len(collision)
total_cas = len(casualty)
fatal_n   = (casualty['severity_label'] == 'Fatal').sum()
serious_n = (casualty['severity_label'] == 'Serious').sum()
slight_n  = (casualty['severity_label'] == 'Slight').sum()

top_weather = collision['weather_label'].value_counts().idxmax()
top_road    = collision['road_cond_label'].value_counts().idxmax()
top_light   = collision['light_label'].value_counts().idxmax()
top_day     = collision['day_label'].value_counts().idxmax()
top_hour    = int(collision['hour'].value_counts().idxmax())
top_cas_type = casualty['cas_type_label'].value_counts().idxmax()
top_veh     = vehicle['veh_type_label'].value_counts().idxmax()

urban_pct = (collision['urban_label'] == 'Urban').sum() / total_col * 100
male_pct  = (casualty['sex_label'] == 'Male').sum() / len(casualty[casualty['sex_label'].isin(['Male','Female'])]) * 100

print(f"\n📊 SCALE")
print(f"   Total Collisions  : {total_col:,}")
print(f"   Total Casualties  : {total_cas:,}")
print(f"\n⚠️  SEVERITY")
print(f"   Fatal    : {fatal_n:,}  ({fatal_n/total_cas*100:.1f}%)")
print(f"   Serious  : {serious_n:,}  ({serious_n/total_cas*100:.1f}%)")
print(f"   Slight   : {slight_n:,}  ({slight_n/total_cas*100:.1f}%)")
print(f"\n🕐 TIMING")
print(f"   Busiest day    : {top_day}")
print(f"   Peak hour      : {top_hour:02d}:00–{top_hour+1:02d}:00")
print(f"\n🌦  CONDITIONS")
print(f"   Most common weather  : {top_weather}")
print(f"   Most common road     : {top_road}")
print(f"   Most common light    : {top_light}")
print(f"\n👥 DEMOGRAPHICS")
print(f"   Urban collisions     : {urban_pct:.1f}%")
print(f"   Male casualties      : {male_pct:.1f}%")
print(f"   Top casualty type    : {top_cas_type}")
print(f"\n🚗 VEHICLES")
print(f"   Most involved type   : {top_veh}")
print("="*65)

In [ ]:
# ── Final Summary Infographic ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(22, 12))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('🚦 UK Road Casualty 2025 — Executive Summary Dashboard',
             fontsize=17, fontweight='bold', color='white', y=1.01)

# 1: severity donut (casualties)
ax = axes[0][0]
sev_cas_vals = [fatal_n, serious_n, slight_n]
sev_labels_d = [f'Fatal\n{fatal_n:,}', f'Serious\n{serious_n:,}', f'Slight\n{slight_n:,}']
wedges, texts = ax.pie(sev_cas_vals, labels=sev_labels_d,
                       colors=[RED, ORANGE, GREEN], startangle=90,
                       wedgeprops={'edgecolor': DARK_BG, 'linewidth': 3, 'width': 0.55},
                       textprops={'color': 'white', 'fontsize': 10, 'fontweight': 'bold'})
ax.text(0, 0, f'{total_cas:,}\nCasualties', ha='center', va='center',
        color='white', fontsize=11, fontweight='bold')
style_ax(ax, 'Casualty Severity Split', '')

# 2: hourly line chart
ax = axes[0][1]
hc = collision['hour'].value_counts().sort_index()
ax.fill_between(hc.index, hc.values, alpha=0.3, color=TEAL)
ax.plot(hc.index, hc.values, color=TEAL, linewidth=2.5, marker='o', markersize=4)
ax.axvspan(7, 9, alpha=0.2, color=ORANGE, label='AM rush')
ax.axvspan(16, 19, alpha=0.2, color=RED, label='PM rush')
ax.legend(fontsize=9)
style_ax(ax, 'Hourly Collision Profile', 'Hour', 'Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# 3: top 5 casualty types
ax = axes[0][2]
top5_ct = casualty['cas_type_label'].value_counts().head(5)
colors_5 = [GREEN, BLUE, TEAL, ORANGE, PURPLE]
bars = ax.barh(top5_ct.index[::-1], top5_ct.values[::-1], color=colors_5[::-1], edgecolor='none')
for bar in bars:
    w = bar.get_width()
    ax.text(w + 50, bar.get_y() + bar.get_height()/2,
            f'{w:,}', va='center', color='white', fontsize=9, fontweight='bold')
style_ax(ax, 'Top 5 Casualty Types', 'Count', '')
ax.set_xlim(0, top5_ct.max() * 1.25)

# 4: road conditions bar
ax = axes[1][0]
rc2 = collision['road_cond_label'].value_counts().head(5)
bars = ax.bar(rc2.index, rc2.values, color=[GREEN, BLUE, ORANGE, RED, PURPLE], edgecolor='none', width=0.6)
for bar, val in zip(bars, rc2.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}', ha='center', va='bottom', color='white', fontsize=9)
ax.set_xticklabels(rc2.index, rotation=20, ha='right')
style_ax(ax, 'Road Surface Conditions', '', 'Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# 5: urban vs rural + severity
ax = axes[1][1]
ur_sev = collision.groupby(['urban_label','severity_label']).size().unstack(fill_value=0)
ur_sev = ur_sev.reindex(index=['Urban','Rural'], columns=['Fatal','Serious','Slight'], fill_value=0)
x = np.arange(2)
w2 = 0.28
for i, (col, col_c) in enumerate(zip(['Fatal','Serious','Slight'], [RED, ORANGE, GREEN])):
    bars = ax.bar(x + i*w2, ur_sev[col], width=w2, color=col_c, label=col)
ax.set_xticks(x + w2)
ax.set_xticklabels(['Urban', 'Rural'])
ax.legend(fontsize=9)
style_ax(ax, 'Urban vs Rural × Severity', '', 'Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# 6: driver age band
ax = axes[1][2]
da2 = vehicle[vehicle['driver_age_band_label'].isin(age_drv_order)]    ['driver_age_band_label'].value_counts().reindex(age_drv_order, fill_value=0)
line_c = [RED if a in ['16-20','21-25'] else BLUE for a in age_drv_order]
bars = ax.bar(age_drv_order, da2.values, color=line_c, edgecolor='none', width=0.7)
ax.set_xticklabels(age_drv_order, rotation=30, ha='right', fontsize=9)
style_ax(ax, 'Driver Age Band (Involved)', 'Age Band', 'Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('11_executive_dashboard.png', bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print("\n✅ All analysis complete! Charts saved as PNG files.")